# Correcting MAD and FF Calculations for Varied Salinity #
Author: Maddox Chastain

Summer 2026

## 1.0 Introduction ##

In [57]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 2.0 Loading in Salinity Data ##

In [58]:
saldf = pd.read_csv("/mnt/c/Users/maddo/onedrive/Desktop/Work stuff/SURF 2026/MAD-FF-Corrections-from-Varied-Salinity/spreadsheets/501-all-water-data_Master_23_03.csv")
saldf = saldf.sort_values(by='Top Depth of Sample [m b.s.f.]')

In [59]:
clean_sal = saldf.loc[(saldf['Contamination'] != 'YY'), 'Salinity [‰]']

clean_dep = saldf.loc[(saldf['Contamination'] != 'YY'), 'Top Depth of Sample [m b.s.f.]']

## 3.0 MAD Calculations with Linear Interpolation ##

### 3.1 Initial Data Loading and Tweaking ###

In [60]:
df = pd.read_csv('/mnt/c/Users/maddo/onedrive/Desktop/Work stuff/SURF 2026/MAD-FF-Corrections-from-Varied-Salinity/spreadsheets/03_MAD/COMBINED_MAD/MAD_DATA_CSV.csv')
bkrs_df = pd.read_csv('/mnt/c/Users/maddo/onedrive/Desktop/Work stuff/SURF 2026/MAD-FF-Corrections-from-Varied-Salinity/spreadsheets/03_MAD/COMBINED_MAD/beakers_CSV.csv')#, index_col = 'Number')

In [61]:
for row in range(0, 600):
    num = df.iat[row, 12]
    for line in range(0, 137):
        if bkrs_df.iat[line, 0] == num:
            value = bkrs_df.iat[line, 2]
            df.at[row, 'Beaker Volume, cc'] = value
        else:
            continue

In [62]:
df = df.sort_values(by='Sample Bot Depth, mbsf')

In [63]:
df['Salinity (‰)'] = np.interp(df['Sample Bot Depth, mbsf'], clean_dep, clean_sal)

In [64]:
df.head(n=4)

,Exp,Site,Hole,Core,Type,Section,"Sample Top, cm","Sample Bottom, cm",Combined id,Sample Label,...,Beaker Number,"Beaker Weight, g","WET Weight (Sample + Beaker), g",OVEN Date/Time IN,"DRY Weight (Sample + Beaker), g",Scientist,Pycnometer Cell,"DRY Volume, cc","Beaker Volume, cc",Salinity (‰)
415,501,M0113,A,1,H,1,38.0,40.0,M0113-A-1-1,"501-M0113A-1H-1,38-40",...,201,9.7538,17.3776,2/2/2026 18:33,15.2597,MS,5,6.45130,4.3739,31.960000
190,501,M0112,A,1,H,1,55.0,57.0,M0112-A-1-1,"501-M0112-A-1-1,55-57",...,166,9.5335,17.3532,1/26/2026 10:00,15.6159,TS,1,6.57320,4.2751,32.048000
0,501,M0111,A,1,H,1,63.5,65.5,M0111-A-1-1,"501-M0111A-1H-1,63.5-65.5",...,280,10.0747,19.1952,1/16/2026 11:00,17.7360,Shintani,1,7.39638,4.5178,32.400231
191,501,M0112,A,2,H,1,38.0,40.0,M0112-A-2-1,"501-M0112-A-2-1,38-40",...,167,9.8196,19.8212,1/26/2026 10:00,17.6881,TS,2,7.38350,4.4034,32.346933


### 3.2 Basic Calculations ###

In [65]:
full_wet_weight = df['WET Weight (Sample + Beaker), g ']
full_dry_weight = df['DRY Weight (Sample + Beaker), g']
beaker_weight = df['Beaker Weight,         g']


# WEIGHT WET Sample, g (full wet weight - beaker weight)
df['WEIGHT WET Sample, g'] = full_wet_weight - beaker_weight
wet_weight = df['WEIGHT WET Sample, g']

# WEIGHT DRY Sample, g (full dry weight - breaker weight)
df['WEIGHT DRY Sample, g'] = full_dry_weight - beaker_weight
dry_weight = df['WEIGHT DRY Sample, g']

# WEIGHT Pore Water, g ( (wet weight - dry weight)/(1-s), where s = 35/1000 or 0.035 )
df['WEIGHT Pore Water, g'] = (wet_weight - dry_weight)/(1-35/1000)

# WEIGHT Salt, g (weight pore water - (wet weight - dry weight))
df['WEIGHT Salt, g'] = (df['WEIGHT Pore Water, g']) - (wet_weight - dry_weight)

# WEIGHT Grains, g (weight dry - weight salt)
df['WEIGHT Grains, g'] = (dry_weight - 0.035*wet_weight)/(1-0.035)

### 3.3 Salinity Sensitive Calculations ###
AKA... where the non-constant salinity actually matters :P

In [66]:
pw_weight = df['WEIGHT Pore Water, g']
salt_weight = df['WEIGHT Salt, g']
dry_volume = df['DRY Volume, cc']
beaker_volume = df['Beaker Volume, cc']

rho_pw = 1.024
rho_salt = 2.22


# VOLUME Pore Water, cc (pore water weight / pore water density) 
df['VOLUME Pore Water, cc'] = pw_weight / rho_pw

# VOLUME Salt, cc (weight salt / density salt)
df['VOLUME Salt, cc'] = salt_weight / rho_salt

# VOLUME Dry Sample, cc (dry volume - beaker volume)
df['VOLUME Dry Sample, cc'] = dry_volume - beaker_volume

# VOLUME Wet Sample, cc (dry sample volume - salt volume + pore water volume)
sample_dry_volume = df['VOLUME Dry Sample, cc']
salt_volume = df['VOLUME Salt, cc']
pw_volume = df['VOLUME Pore Water, cc']

df['VOLUME Wet Sample, cc'] = sample_dry_volume - salt_volume + pw_volume

# VOLUME Grains, cc (dry sample volume - salt volume)
df['VOLUME Grains, cc'] = sample_dry_volume - salt_volume

In [67]:
grain_weight = df['WEIGHT Grains, g']


# Water Content, (wet) v/v (weight pore water / weight wet sample)
df['Water Content, (wet) v/v'] = pw_weight / wet_weight

# Water Content, (dry) v/v (weight pore water / weight grains)
df['Water Content, (dry) v/v'] = pw_weight / grain_weight

In [68]:
sample_wet_volume = df['VOLUME Wet Sample, cc']
grain_weight = df['WEIGHT Grains, g']
grain_volume = df['VOLUME Grains, cc']


# Bulk Density, g/cc (weight wet sample / volume wet sample)
df['Bulk Density, g/cc'] = wet_weight / sample_wet_volume

# Dry Density, g/cc (weight salt / volume wet sample) <- eqn used in spreadsheets
# (weight grains / volume wet sample) <- eqn used in handbook
df['Dry Density, g/cc'] = grain_weight / sample_wet_volume

# Grain Density, g/cc (weight grains / volume grains)
df['Grain Density, g/cc'] = grain_weight / grain_volume

In [69]:
# Porosity, v/v (volume pore water / volume wet sample)
df['Porosity, v/v'] = pw_volume / sample_wet_volume

# Void Ratio, v/v (volume pore water / volume grains)
df['Void Ratio, v/v'] = pw_volume / grain_volume

In [70]:
df.head(n=4)

,Exp,Site,Hole,Core,Type,Section,"Sample Top, cm","Sample Bottom, cm",Combined id,Sample Label,...,"VOLUME Dry Sample, cc","VOLUME Wet Sample, cc","VOLUME Grains, cc","Water Content, (wet) v/v","Water Content, (dry) v/v","Bulk Density, g/cc","Dry Density, g/cc","Grain Density, g/cc","Porosity, v/v","Void Ratio, v/v"
415,501,M0113,A,1,H,1,38.0,40.0,M0113-A-1-1,"501-M0113A-1H-1,38-40",...,2.07740,4.186075,2.042799,0.287877,0.404251,1.821229,1.296939,2.657670,0.512001,1.049186
190,501,M0112,A,1,H,1,55.0,57.0,M0112-A-1-1,"501-M0112-A-1-1,55-57",...,2.29810,4.027833,2.269717,0.230228,0.299085,1.941416,1.494449,2.652044,0.436492,0.774597
0,501,M0111,A,1,H,1,63.5,65.5,M0111-A-1-1,"501-M0111A-1H-1,63.5-65.5",...,2.87858,4.331424,2.854740,0.165794,0.198745,2.105658,1.756553,2.665173,0.340923,0.517274
191,501,M0112,A,2,H,1,38.0,40.0,M0112-A-2-1,"501-M0112-A-2-1,38-40",...,2.98010,5.103909,2.945250,0.221011,0.283716,1.959596,1.526503,2.645321,0.422942,0.732929
